# 13 — Calibración de los modelos por clúster (mecanismo y remedio)

Cierra el argumento de §5.2 con dos pruebas sobre los modelos ya entrenados
(se reutiliza el output del **notebook 12** como kernel source — no se reentrena):

1. **Mecanismo**: si el MBE existe también en TRAIN y VAL (no solo en test), el
   sesgo es una propiedad de calibración de la función aprendida — no un
   artefacto de la partición ni del conjunto de evaluación.
2. **Remedio**: la post-calibración por clúster (restar la constante estimada
   **solo en validación**) corrige el MBE de test → el análisis por clúster
   conserva su valor de diagnóstico territorial sin el costo de equidad.

**Fuentes de datos requeridas:** dataset `lucasiturriago/pruebassaber11` +
notebook `lucasiturriago/12-diagnostico-mbe-saberxai` (aporta `models/`).
**Acelerador:** GPU T4 x2 (`NvidiaTeslaT4`), Internet activado.

In [ ]:
!git clone https://github.com/liturriago/EquidadAlgoritmica-SaberXAI.git
%cd EquidadAlgoritmica-SaberXAI
!pip install -e . -q

In [ ]:
# Resolver dataset y modelos (los modelos vienen del OUTPUT del notebook 12, como kernel source)
from glob import glob
from pathlib import Path
import subprocess

print(subprocess.run(['ls', '/kaggle/input'], capture_output=True, text=True).stdout)

cands = [
    Path('/kaggle/input/pruebassaber11'),
    Path('/kaggle/input/datasets/lucasiturriago/pruebassaber11'),
]
data_dir = next((p for p in cands if (p / 'clase_0.parquet').exists()), None)
if data_dir is None:
    import kagglehub
    data_dir = Path(kagglehub.dataset_download('lucasiturriago/pruebassaber11'))
print('data_dir =', data_dir)

# Búsqueda recursiva: la ruta de montaje de los kernel sources varía entre corridas
hits = sorted(glob('/kaggle/input/**/xgb_cluster_6.json', recursive=True))
assert hits, "No se encontró xgb_cluster_6.json bajo /kaggle/input. Adjunta el notebook 12 como fuente de datos (kernel source)."
models_dir = Path(hits[0]).parent
print('models_dir =', models_dir)

# Helper: ejecuta un comando y aborta el notebook si falla (los !python no abortan)
import shlex
import subprocess

def run(cmd):
    print(">>", cmd)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"Falló (exit {r.returncode}): {cmd}"


In [ ]:
# Calibración: MBE en train/val/test + post-calibración con constante de VAL
run(f"python scripts/evaluar_calibracion.py --config configs/lca_clusters.yaml --model-type both --data-dir {shlex.quote(str(data_dir))} --models-dir {shlex.quote(str(models_dir))}")

In [ ]:
# Tabla resumen
import json
import pandas as pd

with open('results/calibracion_mbe.json') as f:
    cal = json.load(f)

rows = []
for r in cal:
    for mt, rep in r['models'].items():
        rows.append({
            'clúster': r['cluster_id'], 'modelo': mt,
            'MBE_train': round(rep['train']['mbe'], 2),
            'MBE_val': round(rep['val']['mbe'], 2),
            'MBE_test': round(rep['test']['mbe'], 2),
            'constante_val': round(rep['post_calibracion']['constante_val'], 2),
            'MBE_test_corr': round(rep['post_calibracion']['test']['mbe'], 2),
            'RMSE_test_corr': round(rep['post_calibracion']['test']['rmse'], 2),
            'R2_test_corr': round(rep['post_calibracion']['test']['r2'], 4),
        })
pd.DataFrame(rows)